# Capstone Project: A Book Catalogue Pipeline

Nineteen chapters of pieces. This one builds something out of them.

We are going to write a small program that reads a shop's catalogue across several pages, pulls out the fields that matter, throws away the rows that are not usable, and writes a clean dataset to disk. That is not a toy: it is the shape of most of the work that happens before anyone in data science opens a notebook.

**What it does, in one line each:**

1. Walks the catalogue page by page, following the "next" link until there is not one
2. Pulls the fields out of each product card
3. Cleans the messy strings into numbers and booleans
4. Rejects anything that fails validation, and says why
5. Writes `books.csv` and `books.json`, and a summary of what happened

**The chapters it uses:**

| Piece | From |
|---|---|
| A package with modules and `__main__` | 11 |
| A generator that walks the pages | 12 |
| A `@retry` decorator on the loading step | 13 |
| `with` for every file, and a timer | 10, 14 |
| `dataclass`, `Counter`, `datetime` | 8, 15 |
| Regex for prices, stock counts and ratings | 16 |
| Retries with backoff, and the HTTP version in section 11 | 17 |
| BeautifulSoup for the parsing | 18 |
| Type hints, docstrings, logging, tests | 19 |
| Custom exceptions for the rejections | 9 |

**One decision worth stating up front.** The pipeline reads pages through a function you pass in, so this notebook runs it against the HTML committed in `sample_data/`. It needs no network, gives the same answer every time, and works on a train. Section 11 shows the one module you change to make it fetch over HTTP instead, which is exactly what chapter 18 already did live.

---
# 1. The Project Layout

Chapter 11 said a program stops being a script when it grows past one file. This is that point.

```
capstone_demo/
├── bookshop/
│   ├── __init__.py       what the package exposes
│   ├── errors.py         the exceptions this package raises
│   ├── models.py         what a Book is, and what makes one valid
│   ├── clean.py          messy strings to real values
│   ├── parse.py          HTML to raw fields
│   ├── pipeline.py       walking the pages, building the catalogue
│   ├── storage.py        writing CSV and JSON
│   └── __main__.py       so `python -m bookshop` runs it
├── tests/
│   ├── test_clean.py
│   └── test_models.py
├── data/                 the pages to read
├── output/               what we produce
└── check_retry.py        a scratch script, not part of the package
```

Each module has one job, and the dependencies only ever point one way: `__main__` uses `pipeline`, `pipeline` uses `parse` and `models`, `models` uses `clean` and `errors`. Nothing points back up. That is what makes the pieces testable on their own.

In [1]:
import os
import shutil
from pathlib import Path

ROOT = Path("capstone_demo")

shutil.rmtree(ROOT, ignore_errors=True)                 # chapter 10's shutil
for folder in ["bookshop", "tests", "data", "output"]:
    (ROOT / folder).mkdir(parents=True)

for page in range(1, 4):                                # the pages to read
    shutil.copy(f"sample_data/shop_page_{page}.html", ROOT / "data")

print(sorted(p.name for p in ROOT.iterdir()))
print(sorted(p.name for p in (ROOT / "data").iterdir()))

['bookshop', 'data', 'output', 'tests']
['shop_page_1.html', 'shop_page_2.html', 'shop_page_3.html']


---
# 2. The Exceptions

Chapter 9 argued for exception classes of your own, so a caller can catch the failure it knows how to handle. Three is enough here, and they share a base class so anything using this package can catch `BookshopError` and be sure it has caught everything the package throws.

In [2]:
%%writefile capstone_demo/bookshop/errors.py
"""Everything this package raises."""


class BookshopError(Exception):
    """Base class, so a caller can catch everything from this package."""


class PageUnavailable(BookshopError):
    """A page could not be read."""


class InvalidBook(BookshopError):
    """A product card was parsed but is not usable."""

    def __init__(self, sku: str, reason: str):
        self.sku = sku
        self.reason = reason
        super().__init__(f"{sku}: {reason}")

Writing capstone_demo/bookshop/errors.py


`InvalidBook` carries the sku and the reason as attributes rather than only in the message, so the code that catches it can count reasons rather than parse strings. That distinction is small and it is the difference between a log you can act on and a log you can only read.

---
# 3. Cleaning the Strings

Everything on a web page is a string. `"£51.77"`, `"In stock (22 available)"` and `"rating star-four"` all have to become a number, a boolean with a count, and an integer.

This is chapter 16's work, in its own module because it is the part with rules in it, and therefore the part most worth testing.

In [3]:
%%writefile capstone_demo/bookshop/clean.py
"""Turning the strings on the page into real values."""
import re

PRICE = re.compile(r"(?P<currency>[^\d\s.,]*)\s*(?P<amount>[\d,]+\.?\d*)")
STOCK_COUNT = re.compile(r"\((?P<count>\d+)\s+available\)")

WORD_TO_NUMBER = {"one": 1, "two": 2, "three": 3, "four": 4, "five": 5}


def to_price(text: str | None) -> tuple[str | None, float | None]:
    """Split a price string into its currency symbol and its amount.

    "£1,299.00" becomes ("£", 1299.0). Anything without digits in it
    becomes (None, None).
    """
    if not text:
        return None, None
    found = PRICE.search(text)
    if not found:
        return None, None
    amount = found.group("amount").replace(",", "")
    return found.group("currency") or None, float(amount)


def to_stock(text: str | None) -> tuple[bool, int | None]:
    """Read an availability string.

    "In stock (22 available)" becomes (True, 22), "Out of stock"
    becomes (False, None), and anything unrecognised becomes (False, None).
    """
    if not text or not text.lower().startswith("in stock"):
        return False, None
    found = STOCK_COUNT.search(text)
    return True, int(found.group("count")) if found else None


def to_rating(classes: list[str]) -> int | None:
    """Read a star rating out of a list of CSS classes.

    ["rating", "star-four"] becomes 4. Returns None if no star class is there.
    """
    for name in classes:
        if name.startswith("star-"):
            return WORD_TO_NUMBER.get(name.removeprefix("star-"))
    return None

Writing capstone_demo/bookshop/clean.py


Three functions, each with one job, each returning something the caller can check rather than raising. Nothing here knows about HTML or files, which is why the tests for it in section 9 need no fixtures at all.

Note `to_price` returns a **tuple**. A price of `51.77` is meaningless without knowing it is pounds, and returning both keeps them together.

---
# 4. What a Book Is

Chapter 15 introduced `@dataclass` as the answer to "a namedtuple is not enough". This is the case it was made for: a record with fields, a couple of derived values, and rules about what counts as valid.

Validation happens in `__post_init__`, which a dataclass calls after the fields are set. A book with no price is not an error in the parser, it is a book we cannot use, and saying so precisely is the point of `InvalidBook`.

In [4]:
%%writefile capstone_demo/bookshop/models.py
"""The Book record, and the rules for a usable one."""
from dataclasses import asdict, dataclass

from .errors import InvalidBook


@dataclass
class Book:
    """One product from the catalogue."""

    sku: str
    title: str
    author: str | None
    currency: str | None
    price: float | None
    in_stock: bool
    stock_count: int | None
    rating: int | None

    def __post_init__(self) -> None:
        if not self.sku:
            raise InvalidBook("(no sku)", "the card has no data-sku attribute")
        if not self.title:
            raise InvalidBook(self.sku, "no title")
        if self.price is None:
            raise InvalidBook(self.sku, "no price on the page")
        if self.price <= 0:
            raise InvalidBook(self.sku, f"price is not positive: {self.price}")
        if self.rating is not None and not 1 <= self.rating <= 5:
            raise InvalidBook(self.sku, f"rating out of range: {self.rating}")

    @property
    def value_score(self) -> float | None:
        """Rating per pound, for the cheapest-good-book question."""
        if self.rating is None or not self.price:
            return None
        return round(self.rating / self.price, 4)

    def as_row(self) -> dict:
        """A flat dict, ready for csv.DictWriter or json.dump."""
        row = asdict(self)
        row["value_score"] = self.value_score
        return row

Writing capstone_demo/bookshop/models.py


`@property` is chapter 13, `asdict` comes free with the dataclass, and `InvalidBook` is chapter 9. The rules are all in one place, so there is exactly one answer to "what makes a book usable" and it is readable in ten lines.

Worth noticing: `__post_init__` raises rather than returning a flag. That means a `Book` object either exists and is valid, or does not exist. No code downstream has to wonder.

---
# 5. Reading the Page

Chapter 18's job, and its lesson: every field goes through a helper that copes with the field not being there.

In [5]:
%%writefile capstone_demo/bookshop/parse.py
"""HTML in, raw field dictionaries out."""
from bs4 import BeautifulSoup

from .clean import to_price, to_rating, to_stock


def text_of(parent, selector: str) -> str | None:
    """The first match for selector as tidy text, or None."""
    found = parent.select_one(selector)
    return found.get_text(strip=True) if found else None


def parse_card(article) -> dict:
    """Pull the fields out of one product card."""
    currency, price = to_price(text_of(article, "p.price"))
    in_stock, stock_count = to_stock(text_of(article, "p.stock"))
    rating_tag = article.select_one("p.rating")

    return {
        "sku": article.get("data-sku", ""),
        "title": text_of(article, "h3 a") or "",
        "author": text_of(article, "p.author"),
        "currency": currency,
        "price": price,
        "in_stock": in_stock,
        "stock_count": stock_count,
        "rating": to_rating(rating_tag.get("class", []) if rating_tag else []),
    }


def parse_page(html: str) -> tuple[list[dict], str | None]:
    """Every card on a page, and the link to the next page if there is one."""
    soup = BeautifulSoup(html, "html.parser")
    cards = [parse_card(article) for article in soup.select("article.product")]
    next_link = soup.select_one("nav.pagination a.next")
    return cards, next_link["href"] if next_link else None

Writing capstone_demo/bookshop/parse.py


`parse_page` returns the cards **and** where to go next, because the page is the only thing that knows. The pipeline in the next section does not need to guess how many pages there are or construct URLs.

---
# 6. Walking the Catalogue

The heart of it, and where the most chapters meet. `crawl` is a generator (12), `load_page` is wrapped in a retry decorator (13, 17), and everything that happens gets logged (19).

In [6]:
%%writefile capstone_demo/bookshop/pipeline.py
"""Walking the pages and building the catalogue."""
import functools
import logging
import time
from collections.abc import Callable, Iterator

from .errors import InvalidBook, PageUnavailable
from .models import Book
from .parse import parse_page

logger = logging.getLogger(__name__)


def retry(times: int = 3, delay: float = 0.5):
    """Retry a function that raises PageUnavailable, backing off each time."""
    def decorator(func):
        @functools.wraps(func)
        def wrapper(*args, **kwargs):
            wait = delay
            for attempt in range(1, times + 1):
                try:
                    return func(*args, **kwargs)
                except PageUnavailable as e:
                    if attempt == times:
                        raise
                    logger.warning("attempt %s failed (%s), waiting %ss",
                                   attempt, e, wait)
                    time.sleep(wait)
                    wait *= 2
        return wrapper
    return decorator


def crawl(start: str, load: Callable[[str], str]) -> Iterator[dict]:
    """Yield every product card, following the next link until it runs out."""
    page = start
    while page:
        logger.info("reading %s", page)
        cards, page = parse_page(load(page))
        logger.debug("%s cards on that page", len(cards))
        yield from cards


def build_catalogue(
    start: str, load: Callable[[str], str]
) -> tuple[list[Book], list[InvalidBook]]:
    """Return the books that validated, and the rejections with reasons."""
    books: list[Book] = []
    rejected: list[InvalidBook] = []

    for card in crawl(start, load):
        try:
            books.append(Book(**card))
        except InvalidBook as e:
            logger.warning("rejected %s", e)
            rejected.append(e)

    logger.info("kept %s books, rejected %s", len(books), len(rejected))
    return books, rejected

Writing capstone_demo/bookshop/pipeline.py


Three things worth pausing on.

`crawl` **yields cards, not pages**, so the caller never sees pagination at all. Chapter 12's point, applied.

`load` is a **parameter**, typed `Callable[[str], str]`. The pipeline does not know or care whether pages come from a disk or a web server, which is what makes it testable, and what lets section 11 move the whole project onto the network by rewriting one function.

`build_catalogue` returns **both** the successes and the failures. A pipeline that silently drops bad rows is how you end up with a dataset that is quietly missing a tenth of its data.

### Does the retry actually work?

`retry` is on `load_page` in section 8, and reading a file that is sitting right there will never fail, so nothing in this project as it stands will ever make it fire. A decorator you have never seen fire is a decorator you do not know works, so here is a scratch script that fails twice and then succeeds.

The `run` helper below is what this notebook uses to run the project in a subprocess. As in chapter 19, it blanks out the duration so the committed output stays the same on every run.

In [7]:
%%writefile capstone_demo/check_retry.py
"""Prove the retry decorator does what section 6 claims."""
import logging
import sys

from bookshop.errors import PageUnavailable
from bookshop.pipeline import retry

logging.basicConfig(level=logging.INFO,
                    format="%(levelname)-8s %(name)s | %(message)s",
                    stream=sys.stdout)

attempts = {"count": 0}


@retry(times=3, delay=0.1)
def flaky_page() -> str:
    attempts["count"] += 1
    if attempts["count"] < 3:
        raise PageUnavailable(
            f"connection reset on attempt {attempts['count']}")
    return "<html>the page, finally</html>"


print("got:", flaky_page())
print("after", attempts["count"], "attempts")


@retry(times=2, delay=0.1)
def always_missing() -> str:
    raise PageUnavailable("no such page: shop_page_99.html")


try:
    always_missing()
except PageUnavailable as e:
    print("gave up, and said so:", e)

Writing capstone_demo/check_retry.py


In [8]:
import re
import subprocess
import sys


def run(*args, cwd="capstone_demo"):
    result = subprocess.run([sys.executable, *args], cwd=cwd,
                            capture_output=True, text=True,
                            env={**os.environ, "COLUMNS": "80", "PYTHONPATH": "."})
    # pytest and the pipeline both print a duration, which changes every run.
    # "waiting 0.1s" is deliberately not of that shape, so the backoff stays visible.
    print(re.sub(r"in \d+\.\d+s", "in Xs", result.stdout + result.stderr))
    return result.returncode


print("exit code:", run("check_retry.py"))

WARNING  bookshop.pipeline | attempt 1 failed (connection reset on attempt 1), waiting 0.1s
WARNING  bookshop.pipeline | attempt 2 failed (connection reset on attempt 2), waiting 0.2s
got: <html>the page, finally</html>
after 3 attempts
WARNING  bookshop.pipeline | attempt 1 failed (no such page: shop_page_99.html), waiting 0.1s
gave up, and said so: no such page: shop_page_99.html

exit code: 0


Two warnings, then success on the third attempt, with the wait doubling from 0.1 to 0.2 seconds. And when the page is never going to arrive, the exception comes through rather than being swallowed, which is what chapter 14 argued for and what stops a pipeline reporting a clean run over missing data.

---
# 7. Writing the Output

Chapter 10, with chapter 14's `with` on every file.

In [9]:
%%writefile capstone_demo/bookshop/storage.py
"""Writing the catalogue out."""
import csv
import json
import logging
from datetime import datetime, timezone
from pathlib import Path

from .models import Book

logger = logging.getLogger(__name__)

FIELDS = ["sku", "title", "author", "currency", "price",
          "in_stock", "stock_count", "rating", "value_score"]


def write_csv(books: list[Book], path: Path) -> None:
    """Write the books as CSV, one row each, with a header."""
    with open(path, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=FIELDS)
        writer.writeheader()
        writer.writerows(book.as_row() for book in books)
    logger.info("wrote %s rows to %s", len(books), path)


def write_json(books: list[Book], path: Path, source: str) -> None:
    """Write the books as JSON, with a little about where they came from."""
    document = {
        "generated_at": datetime.now(timezone.utc).isoformat(
            timespec="seconds"),
        "source": source,
        "count": len(books),
        "books": [book.as_row() for book in books],
    }
    with open(path, "w", encoding="utf-8") as f:
        json.dump(document, f, indent=2, ensure_ascii=False)
    logger.info("wrote %s books to %s", len(books), path)

Writing capstone_demo/bookshop/storage.py


The JSON file records **when** it was made and **where the data came from**. Six months later, that header is the difference between a dataset you can trust and a file called `books_final_v2.json` that nobody dares use.

`ensure_ascii=False` keeps the `£` and the accented names readable rather than escaped, which chapter 10 mentioned and chapter 18's mojibake made vivid.

---
# 8. The Entry Point

Chapter 11's `if __name__ == "__main__"`, made real. `__main__.py` inside a package is what makes `python -m bookshop` work.

In [10]:
%%writefile capstone_demo/bookshop/__init__.py
"""A small pipeline that turns a shop's catalogue pages into a dataset."""
from .models import Book
from .pipeline import build_catalogue

__all__ = ["Book", "build_catalogue"]
__version__ = "1.0.0"

Writing capstone_demo/bookshop/__init__.py


In [11]:
%%writefile capstone_demo/bookshop/__main__.py
"""Run the whole pipeline: python -m bookshop"""
import logging
import sys
import time
from collections import Counter
from pathlib import Path

from .errors import PageUnavailable
from .pipeline import build_catalogue, retry
from .storage import write_csv, write_json

DATA = Path("data")
OUTPUT = Path("output")
START_PAGE = "shop_page_1.html"

logger = logging.getLogger(__name__)


@retry(times=3, delay=0.2)
def load_page(name: str) -> str:
    """Read one page. Raises PageUnavailable if it is not there."""
    path = DATA / name
    if not path.exists():
        raise PageUnavailable(f"no such page: {name}")
    return path.read_text(encoding="utf-8")


def main() -> int:
    """Build the catalogue and write it out. Returns a process exit code."""
    logging.basicConfig(
        level=logging.INFO,
        format="%(levelname)-8s %(name)s | %(message)s",
        stream=sys.stdout,
    )

    started = time.perf_counter()
    OUTPUT.mkdir(exist_ok=True)

    try:
        books, rejected = build_catalogue(START_PAGE, load_page)
    except PageUnavailable as e:
        logger.error("could not start: %s", e)
        return 1

    if not books:
        logger.error("no usable books found")
        return 1

    write_csv(books, OUTPUT / "books.csv")
    write_json(books, OUTPUT / "books.json", source=str(DATA))

    print()
    print(f"{len(books)} books kept, {len(rejected)} rejected")
    for reason, count in Counter(e.reason for e in rejected).most_common():
        print(f"  {count} x {reason}")

    in_stock = sum(1 for b in books if b.in_stock)
    print(f"{in_stock} of {len(books)} in stock")
    print(f"finished in {time.perf_counter() - started:.1f}s")
    return 0


if __name__ == "__main__":
    sys.exit(main())

Writing capstone_demo/bookshop/__main__.py


`main()` **returns an exit code** rather than calling `sys.exit` itself. That keeps it callable from a test or another script, and the `if __name__` block at the bottom is the only place that touches the process.

The `Counter` of rejection reasons is three lines and it is the most useful thing in the output. "8 books kept, 1 rejected" tells you something is wrong; "1 x no price on the page" tells you what.

---
# 9. The Tests

Chapter 19's rule: test the parts with rules in them. That is `clean.py` and `models.py`. Neither needs a file, a network, or a fixture, because neither knows anything about where the data came from.

In [12]:
%%writefile capstone_demo/tests/test_clean.py
"""Tests for the string cleaners."""
import pytest

from bookshop.clean import to_price, to_rating, to_stock


@pytest.mark.parametrize("text, expected", [
    ("£51.77", ("£", 51.77)),
    ("£1,299.00", ("£", 1299.0)),
    ("51.77", (None, 51.77)),
    ("USD 45", ("USD", 45.0)),
    ("", (None, None)),
    ("free", (None, None)),
    (None, (None, None)),
])
def test_to_price(text, expected):
    assert to_price(text) == expected


@pytest.mark.parametrize("text, expected", [
    ("In stock (22 available)", (True, 22)),
    ("In stock", (True, None)),
    ("Out of stock", (False, None)),
    ("", (False, None)),
    (None, (False, None)),
])
def test_to_stock(text, expected):
    assert to_stock(text) == expected


@pytest.mark.parametrize("classes, expected", [
    (["rating", "star-four"], 4),
    (["rating", "star-one"], 1),
    (["rating"], None),
    ([], None),
    (["rating", "star-eleven"], None),
])
def test_to_rating(classes, expected):
    assert to_rating(classes) == expected

Writing capstone_demo/tests/test_clean.py


In [13]:
%%writefile capstone_demo/tests/test_models.py
"""Tests for the Book rules."""
import pytest

from bookshop.errors import InvalidBook
from bookshop.models import Book

GOOD = {
    "sku": "9781617294433",
    "title": "Deep Learning with Python",
    "author": "François Chollet",
    "currency": "£",
    "price": 51.77,
    "in_stock": True,
    "stock_count": 22,
    "rating": 4,
}


def test_a_good_book_is_accepted():
    book = Book(**GOOD)
    assert book.title == "Deep Learning with Python"
    assert book.value_score == round(4 / 51.77, 4)


def test_value_score_is_none_without_a_rating():
    assert Book(**{**GOOD, "rating": None}).value_score is None


@pytest.mark.parametrize("field, value, reason", [
    ("sku", "", "data-sku"),
    ("title", "", "no title"),
    ("price", None, "no price"),
    ("price", -1.0, "not positive"),
    ("rating", 9, "out of range"),
])
def test_bad_books_are_rejected(field, value, reason):
    with pytest.raises(InvalidBook, match=reason):
        Book(**{**GOOD, field: value})


def test_the_rejection_says_which_book():
    with pytest.raises(InvalidBook) as caught:
        Book(**{**GOOD, "price": None})
    assert caught.value.sku == GOOD["sku"]

Writing capstone_demo/tests/test_models.py


`test_bad_books_are_rejected` is five tests in one, and each names the field it breaks. That is chapter 13's decorator and chapter 19's `parametrize`, doing the job they exist for.

Now run them, with the same `run` helper from section 6.

In [14]:
print("exit code:", run("-m", "pytest", "tests", "-q", "--no-header",
                        "--color=no", "-p", "no:cacheprovider"))

.........................                                                [100%]
25 passed in Xs

exit code: 0


---
# 10. Running It

Everything is in place. This is what a user of the project types.

In [15]:
exit_code = run("-m", "bookshop")

print("exit code:", exit_code)

INFO     bookshop.pipeline | reading shop_page_1.html
INFO     bookshop.pipeline | reading shop_page_2.html
WARNING  bookshop.pipeline | rejected 9781098139482: no price on the page
INFO     bookshop.pipeline | reading shop_page_3.html
INFO     bookshop.pipeline | kept 8 books, rejected 1
INFO     bookshop.storage | wrote 8 rows to output/books.csv
INFO     bookshop.storage | wrote 8 books to output/books.json

8 books kept, 1 rejected
  1 x no price on the page
6 of 8 in stock
finished in Xs

exit code: 0


Read the log from the top. Three pages read, one card rejected with the reason attached, the counts written out, and both files produced. The one rejection is the book on page 2 with no price, which chapter 18 first ran into and chapter 19 wrote a test for.

An exit code of `0` means success, which is what a scheduler or a CI job checks.

The `finished in Xs` line is the notebook hiding the duration again, for the same reason it did with `pytest`. Run it yourself and you will see a real number there.

In [16]:
print(Path("capstone_demo/output/books.csv").read_text(encoding="utf-8"))

sku,title,author,currency,price,in_stock,stock_count,rating,value_score
9781617294433,Deep Learning with Python,François Chollet,£,51.77,True,22,4,0.0773
9781593279288,Python Crash Course,Eric Matthes,£,23.99,True,8,5,0.2084
9781492056355,Fluent Python,Luciano Ramalho,£,44.5,False,,5,0.1124
9781593279929,Automate the Boring Stuff,Al Sweigart,£,27.95,True,13,4,0.1431
9781492041139,Data Science from Scratch,Joel Grus,£,41.05,True,11,4,0.0974
9781449373320,Designing Data-Intensive Applications,Martin Kleppmann,£,55.0,True,5,5,0.0909
9780135957059,The Pragmatic Programmer,Dave Thomas,£,39.99,True,17,5,0.125
9781617292231,Grokking Algorithms,Aditya Bhargava,£,31.4,False,,4,0.1274



In [17]:
import json

document = json.loads(Path("capstone_demo/output/books.json").read_text(encoding="utf-8"))

print("keys      :", list(document))
print("source    :", document["source"])
print("count     :", document["count"])
print("first book:")
for field, value in document["books"][0].items():
    print(f"    {field:<12} {value}")

keys      : ['generated_at', 'source', 'count', 'books']
source    : data
count     : 8
first book:
    sku          9781617294433
    title        Deep Learning with Python
    author       François Chollet
    currency     £
    price        51.77
    in_stock     True
    stock_count  22
    rating       4
    value_score  0.0773


Eight rows and nine columns. The only blanks are `stock_count` on the two books that are out of stock, and that is information rather than a gap: there is no count because there is no stock. Every other cell has a value, and every column holds one kind of thing.

The JSON version carries the same data plus a note about where it came from. `generated_at` is not printed above because it is a timestamp, and it would differ every time you ran this.

---
# 11. Making It Fetch Over HTTP

The pipeline reads pages through whatever function you hand it, so pointing it at a website changes one module. `load_page` becomes:

```python
import requests

from .errors import PageUnavailable

SESSION = requests.Session()
SESSION.headers.update({"User-Agent": "bookshop/1.0 (you@example.com)"})


@retry(times=3, delay=1.0)
def load_page(url: str) -> str:
    """Fetch one page. Raises PageUnavailable so the retry can see it."""
    try:
        response = SESSION.get(url, timeout=10)
        response.raise_for_status()
    except requests.exceptions.RequestException as e:
        raise PageUnavailable(str(e)) from e

    response.encoding = "utf-8"          # chapter 18's mojibake
    time.sleep(2)                        # chapter 18's politeness
    return response.text
```

Nothing else changes. `crawl`, `build_catalogue`, `parse_page`, `Book` and the tests are all untouched, because none of them ever knew where the HTML came from.

Three things in those fifteen lines are the whole of chapters 17 and 18: a `timeout` so it cannot hang, `raise_for_status` so a 404 does not become a `KeyError` a hundred lines later, and the network error wrapped in the package's own exception so `retry` can tell "the server is down" apart from "this book has no price".

This notebook does not run that version, on purpose. A capstone you can run offline, twice, and get the same answer is worth more than one that depends on a website being up. Chapter 18 ran the live version, and section 12 says what to point this at.

---
# 12. Where This Goes Next

`books.csv` is the handoff. It is tidy: one row per book, one column per field, no merged cells, no footnotes, no units buried in the values. That is the format every analysis tool wants, and producing it is the job this course was preparing you for.

The next repository picks it up on the first line:

```python
import pandas as pd

books = pd.read_csv("capstone_demo/output/books.csv")

books.describe()                                   # the numbers, summarised
books.groupby("rating")["price"].mean()            # average price by rating
books.sort_values("value_score", ascending=False)  # best rating per pound
books.plot.scatter(x="price", y="rating")          # and a picture
```

Every one of those lines works because the pipeline did its job first: `price` is a number rather than `"£51.77"`, `in_stock` is a boolean rather than `"In stock (22 available)"`, and there are no rows with a missing price because they were rejected and counted.

**pandas is deliberately not installed in this repository.** It belongs to the repository that covers NumPy, pandas and exploratory analysis, and that is where you go next. This one has done what it set out to do: get you to the point where a clean dataset exists and you know exactly how it was made.

### If you want to keep going with this project

In rough order of how much you will learn:

1. **Point it at a real site.** Section 11 has the code. Check `robots.txt` first, as chapter 18 insisted.
2. **Add a command line.** `argparse` turns the constants at the top of `__main__.py` into `python -m bookshop --pages 5 --output data/`.
3. **Cache the pages.** Save each page to disk on the way through, and skip fetching one you already have. Chapter 17 wrote that in three lines.
4. **Write the rejections to their own file.** You have the reasons; a `rejected.csv` makes the data quality visible to whoever uses the dataset.
5. **Add a second source.** A second `parse.py` with different selectors, feeding the same `Book`. This is where a clean layout starts paying you back.
6. **Run it on a schedule** and keep the dated outputs. That is a time series, and time series are where the interesting questions live.

---
# 13. What You Now Know

Twenty chapters, and the capstone used essentially all of them.

| | Chapter | What the capstone did with it |
|---|---|---|
| Values and types | 1 | Every field has a type, and the right one |
| Lists, tuples, strings | 2 | Rows, records, and all the parsing |
| Sets and dictionaries | 3 | `WORD_TO_NUMBER`, every parsed card |
| Operators | 4 | `1 <= rating <= 5` |
| Conditionals and loops | 5 | Every validation rule |
| Functions | 6 | Twelve of them, each with one job |
| Types of functions | 7 | `load` passed in as an argument |
| OOP | 8 | `Book`, and the exception hierarchy |
| Exceptions | 9 | `InvalidBook` carrying a reason |
| Files | 10 | CSV and JSON, read and written |
| Modules and venvs | 11 | The package, and `python -m bookshop` |
| Iterators and generators | 12 | `crawl` yielding cards, not pages |
| Closures and decorators | 13 | `@retry`, `@property`, `@parametrize` |
| Context managers | 14 | `with` on every file |
| Standard library | 15 | `Counter`, `dataclass`, `datetime`, `pathlib` |
| Regular expressions | 16 | Prices, stock counts, star ratings |
| APIs | 17 | Retries, backoff, timeouts, sessions |
| Web scraping | 18 | The parser, and the politeness |
| Reliable code | 19 | Hints, docstrings, logging, 20 tests |

If you started this course unable to write a `for` loop, you can now build a program that runs unattended, fails loudly, explains itself, and produces something another person can use. That is not a beginner's skill any more.

**Where to go from here**

1. **NumPy, pandas and exploratory analysis** are the next repository, and `books.csv` is the file you walk in with.
2. **Then machine learning**, which is a separate repository again: scikit-learn, evaluation, pipelines, and the MLOps and AI engineering that follow.
3. **Meanwhile, build something small of your own.** Not a tutorial project. Something you actually want to exist. That is the only step on this list that cannot be skipped.

Thank you for reading. Go and build the thing.